In [ ]:
import random, copy, math, time
from typing import Dict, List, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

# ══════════════════════════════════════════════════════════════════════
#  1. DATA
# ══════════════════════════════════════════════════════════════════════

def cybershake_50() -> Dict:
    tasks = {}
    tasks["T0"] = {"duration": 3600, "deps": [], "data": {}}
    dur_lv1  = [1200,980,1540,870,1100,1350,760,1420,990,1180,830,1260,1070,940,1310,1050]
    data_lv1 = [120,95,145,88,110,132,76,138,97,115,82,128,104,93,125,103]
    for i in range(16):
        tasks[f"T1_{i}"] = {"duration": dur_lv1[i], "deps": ["T0"], "data": {"T0": data_lv1[i]}}
    dur_lv2  = [320,280,410,195,350,290,230,380,265,315,200,360,245,270,400,310]
    data_lv2 = [45,38,58,27,49,41,32,54,37,44,28,51,35,39,57,43]
    for i in range(16):
        p = f"T1_{i}"
        tasks[f"T2_{i}"] = {"duration": dur_lv2[i], "deps": [p], "data": {p: data_lv2[i]}}
    dur_lv3  = [180,150,220,110,195,160,130,210,145,175,115,200,140,155,215,170]
    data_lv3 = [18,15,22,11,19,16,13,21,14,17,11,20,14,15,21,17]
    for i in range(16):
        p = f"T2_{i}"
        tasks[f"T3_{i}"] = {"duration": dur_lv3[i], "deps": [p], "data": {p: data_lv3[i]}}
    parents = [f"T3_{i}" for i in range(16)]
    tasks["T4"] = {"duration": 600, "deps": parents, "data": {p: 8 for p in parents}}
    return tasks

def vms_5() -> Dict:
    return {
        "VM1": {"speed": 1.0, "cost": 0.10, "power": 100},
        "VM2": {"speed": 1.8, "cost": 0.20, "power": 160},
        "VM3": {"speed": 2.5, "cost": 0.35, "power": 220},
        "VM4": {"speed": 0.7, "cost": 0.06, "power":  70},
        "VM5": {"speed": 3.2, "cost": 0.50, "power": 300},
    }

# ══════════════════════════════════════════════════════════════════════
#  2. TOPOLOGICAL LEVELS
# ══════════════════════════════════════════════════════════════════════

def compute_levels(tasks):
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    level = {t: 0 for t in tasks}
    queue = [t for t in tasks if in_deg[t] == 0]
    while queue:
        node = queue.pop(0)
        for s in succ[node]:
            level[s] = max(level[s], level[node] + 1)
            in_deg[s] -= 1
            if in_deg[s] == 0: queue.append(s)
    return level

def build_groups(tasks):
    level  = compute_levels(tasks)
    max_lv = max(level.values())
    groups = [[] for _ in range(max_lv + 1)]
    for t, lv in level.items(): groups[lv].append(t)
    for g in groups: g.sort()
    return groups

def groups_to_order(groups):
    return [t for g in groups for t in g]

# ══════════════════════════════════════════════════════════════════════
#  3. EVALUATION
# ══════════════════════════════════════════════════════════════════════

def evaluate(order, vm_map, tasks, vms):
    ft = {}; st = {}
    vm_avail = {vm: 0.0 for vm in vms}
    cost = energy = 0.0
    for task in order:
        vn  = vm_map[task]; vm  = vms[vn]
        dur = tasks[task]["duration"] / vm["speed"]
        rdy = 0.0
        for dep in tasks[task]["deps"]:
            f = ft[dep]
            if vm_map[dep] != vn: f += tasks[task]["data"].get(dep, 0) * 0.01
            rdy = max(rdy, f)
        ts = max(rdy, vm_avail[vn]); te = ts + dur
        st[task] = ts; ft[task] = te
        vm_avail[vn] = te
        cost   += dur / 3600 * vm["cost"]
        energy += dur * vm["power"] / 1000
    return (max(ft.values()), cost, energy, {"start_time": st, "finish_time": ft})

# ══════════════════════════════════════════════════════════════════════
#  4. NORMALIZATION & DYNAMIC WEIGHTS
# ══════════════════════════════════════════════════════════════════════

def compute_ref(tasks, vms, groups_ref, n_samples=300):
    vm_names = list(vms.keys())
    all_m, all_c, all_e = [], [], []
    for vm in vm_names:
        order  = groups_to_order(groups_ref)
        vm_map = {t: vm for t in tasks}
        m, c, e, _ = evaluate(order, vm_map, tasks, vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    rng = random.Random(0)
    for _ in range(n_samples):
        rg     = [rng.sample(g, len(g)) for g in groups_ref]
        order  = groups_to_order(rg)
        vm_map = {t: rng.choice(vm_names) for t in tasks}
        m, c, e, _ = evaluate(order, vm_map, tasks, vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    ref = {"m_min": min(all_m), "m_max": max(all_m),
           "c_min": min(all_c), "c_max": max(all_c),
           "e_min": min(all_e), "e_max": max(all_e)}
    for k in ["m","c","e"]:
        if ref[f"{k}_max"] - ref[f"{k}_min"] < 1e-9:
            ref[f"{k}_max"] = ref[f"{k}_min"] + 1.0
    return ref

def normalize_obj(m, c, e, ref):
    f1 = (m-ref["m_min"])/(ref["m_max"]-ref["m_min"]) + 1e-6
    f2 = (c-ref["c_min"])/(ref["c_max"]-ref["c_min"]) + 1e-6
    f3 = (e-ref["e_min"])/(ref["e_max"]-ref["e_min"]) + 1e-6
    return f1, f2, f3

# ══════════════════════════════════════════════════════════════════════
#  5. HEFT
# ══════════════════════════════════════════════════════════════════════

def compute_rank_u(tasks, vms):
    succ = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    n = len(vms); rank_u = {}
    def _r(t):
        if t in rank_u: return rank_u[t]
        w = sum(tasks[t]["duration"]/v["speed"] for v in vms.values()) / n
        rank_u[t] = w if not succ[t] else w + max(
            tasks[s]["data"].get(t,0)*0.01*(n-1)/n + _r(s) for s in succ[t])
        return rank_u[t]
    for t in tasks: _r(t)
    return rank_u

def heft_schedule(tasks, vms):
    rank_u = compute_rank_u(tasks, vms)
    order  = sorted(tasks.keys(), key=lambda t: rank_u[t], reverse=True)
    vm_avail = {vm: 0.0 for vm in vms}
    ft = {}; st = {}; vm_map = {}
    for task in order:
        best_vm = None; best_eft = float("inf"); best_ast = 0.0
        for vn, vm in vms.items():
            dur = tasks[task]["duration"] / vm["speed"]; rdy = 0.0
            for dep in tasks[task]["deps"]:
                f = ft[dep]
                if vm_map[dep] != vn: f += tasks[task]["data"].get(dep,0)*0.01
                rdy = max(rdy, f)
            ast = max(rdy, vm_avail[vn]); eft = ast + dur
            if eft < best_eft: best_eft, best_vm, best_ast = eft, vn, ast
        vm_map[task] = best_vm
        st[task] = best_ast; ft[task] = best_eft
        vm_avail[best_vm] = best_eft
    level  = compute_levels(tasks); max_lv = max(level.values())
    groups = [[] for _ in range(max_lv+1)]
    for t in order: groups[level[t]].append(t)
    return groups, vm_map

def critical_path(groups, vm_map, tasks, vms):
    order = groups_to_order(groups)
    _, _, _, sched = evaluate(order, vm_map, tasks, vms)
    ft_ = sched["finish_time"]; st_ = sched["start_time"]
    mksp = max(ft_.values())
    succ = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    lft = {}
    for t in reversed(order):
        lft[t] = mksp if not succ[t] else min(st_[s] for s in succ[t])
    return [t for t in order if abs(lft[t]-ft_[t]) < 1e-6]

def heft_local_search(groups, vm_map, tasks, vms):
    cp = set(critical_path(groups, vm_map, tasks, vms))
    new_vm = copy.deepcopy(vm_map)
    order  = groups_to_order(groups)
    idx_map = {t: i for i, t in enumerate(order)}
    for task in order:
        if task not in cp: continue
        _, _, _, sched = evaluate(order, new_vm, tasks, vms)
        ft_ = sched["finish_time"]
        best_vm = new_vm[task]; best_eft = float("inf")
        for vn, vm in vms.items():
            dur = tasks[task]["duration"]/vm["speed"]; rdy = 0.0
            for dep in tasks[task]["deps"]:
                f = ft_[dep]
                if new_vm[dep] != vn: f += tasks[task]["data"].get(dep,0)*0.01
                rdy = max(rdy, f)
            busy = max((ft_[t2] for t2 in order if t2 != task and
                        new_vm[t2]==vn and idx_map[t2]<idx_map[task]), default=0.0)
            eft = max(rdy, busy) + dur
            if eft < best_eft: best_eft, best_vm = eft, vn
        new_vm[task] = best_vm
    return groups, new_vm

# ══════════════════════════════════════════════════════════════════════
#  6. COMMON NSGA-II CORE
# ══════════════════════════════════════════════════════════════════════

def dominates(a, b):
    better = False
    for x, y in zip(a, b):
        if x > y: return False
        if x < y: better = True
    return better

def fast_non_dominated_sort(objectives):
    n = len(objectives); cnt = [0]*n; dom = [[] for _ in range(n)]; fronts = [[]]
    for i in range(n):
        for j in range(n):
            if i==j: continue
            if dominates(objectives[i], objectives[j]): dom[i].append(j)
            elif dominates(objectives[j], objectives[i]): cnt[i] += 1
        if cnt[i]==0: fronts[0].append(i)
    k = 0
    while fronts[k]:
        nxt = []
        for i in fronts[k]:
            for j in dom[i]:
                cnt[j] -= 1
                if cnt[j]==0: nxt.append(j)
        k += 1; fronts.append(nxt)
    return [f for f in fronts if f]

def crowding_distance_standard(front_idx, objectives):
    n = len(front_idx); dist = {i: 0.0 for i in front_idx}
    if n <= 2:
        for i in front_idx: dist[i] = float("inf")
        return dist
    for m in range(len(objectives[0])):
        sf  = sorted(front_idx, key=lambda i: objectives[i][m])
        mn  = objectives[sf[0]][m]; mx = objectives[sf[-1]][m]
        rng = mx - mn + 1e-9
        dist[sf[0]] = dist[sf[-1]] = float("inf")
        for k in range(1, n-1):
            dist[sf[k]] += (objectives[sf[k+1]][m] - objectives[sf[k-1]][m]) / rng
    return dist

def crowding_distance_weighted(front_idx, objectives, ref):
    n = len(front_idx); dist = {i: 0.0 for i in front_idx}
    if n <= 2:
        for i in front_idx: dist[i] = float("inf")
        return dist
    all_f  = [normalize_obj(*objectives[i], ref) for i in front_idx]
    avg_f1 = sum(f[0] for f in all_f)/n
    avg_f2 = sum(f[1] for f in all_f)/n
    avg_f3 = sum(f[2] for f in all_f)/n
    denom  = avg_f1*avg_f2 + avg_f1*avg_f3 + avg_f2*avg_f3 + 1e-9
    obj_w  = [(avg_f2*avg_f3)/denom, (avg_f1*avg_f3)/denom, (avg_f1*avg_f2)/denom]
    for m, w_m in enumerate(obj_w):
        sf  = sorted(front_idx, key=lambda i: objectives[i][m])
        mn  = objectives[sf[0]][m]; mx = objectives[sf[-1]][m]
        rng = mx - mn + 1e-9
        dist[sf[0]] = dist[sf[-1]] = float("inf")
        for k in range(1, n-1):
            dist[sf[k]] += w_m*(objectives[sf[k+1]][m]-objectives[sf[k-1]][m])/rng
    return dist

def binary_tournament(pop_size, ranks, distances):
    i, j = random.sample(range(pop_size), 2)
    if   ranks[i] < ranks[j]:                        return i
    elif ranks[j] < ranks[i]:                        return j
    elif distances.get(i,0) >= distances.get(j,0):   return i
    else:                                             return j

def select_survivors(combined, combined_obj, pop_size, crowding_fn, ref=None):
    all_fronts = fast_non_dominated_sort(combined_obj)
    new_pop = []; new_obj = []
    for front in all_fronts:
        if len(new_pop)+len(front) <= pop_size:
            for idx in front:
                new_pop.append(combined[idx]); new_obj.append(combined_obj[idx])
        else:
            remaining = pop_size - len(new_pop)
            cd = (crowding_fn(front, combined_obj, ref) if ref is not None
                  else crowding_fn(front, combined_obj))
            sf = sorted(front, key=lambda i: cd.get(i,0), reverse=True)
            for idx in sf[:remaining]:
                new_pop.append(combined[idx]); new_obj.append(combined_obj[idx])
            break
    return new_pop, new_obj

# ══════════════════════════════════════════════════════════════════════
#  7. ALGO 1 — SIMPLE GA NSGA-II
# ══════════════════════════════════════════════════════════════════════

def random_topo_order(tasks):
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    ready = [t for t in tasks if in_deg[t]==0]; order = []
    while ready:
        t = random.choice(ready); ready.remove(t); order.append(t)
        for s in succ[t]:
            in_deg[s] -= 1
            if in_deg[s]==0: ready.append(s)
    return order

def crossover_order_simple(o1, o2, tasks):
    n = len(o1); a, b = sorted(random.sample(range(n),2)); seg = set(o1[a:b+1])
    prio = {}
    for rank, t in enumerate(o1):
        prio[t] = rank if t in seg else (n + next(i for i,x in enumerate(o2) if x==t))
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    ready  = sorted([t for t in tasks if in_deg[t]==0], key=lambda t: prio[t])
    result = []
    while ready:
        node = ready.pop(0); result.append(node)
        for s in succ[node]:
            in_deg[s] -= 1
            if in_deg[s]==0:
                ready.append(s); ready.sort(key=lambda x: prio[x])
    return result

def mutate_order_simple(order, tasks, rate):
    new_order = order[:]; n = len(new_order)
    for _ in range(n):
        if random.random() < rate:
            i, j = random.randint(0,n-1), random.randint(0,n-1)
            if i==j: continue
            new_order[i], new_order[j] = new_order[j], new_order[i]
            pos = {t: k for k,t in enumerate(new_order)}
            valid = all(pos[dep]<pos[t] for t in new_order for dep in tasks[t]["deps"])
            if not valid: new_order[i], new_order[j] = new_order[j], new_order[i]
    return new_order

def repair_order_kahn(order, tasks):
    in_deg = {t: len(tasks[t]["deps"]) for t in tasks}
    succ   = {t: [] for t in tasks}
    for t in tasks:
        for dep in tasks[t]["deps"]: succ[dep].append(t)
    priority = {t: i for i,t in enumerate(order)}
    ready = sorted([t for t in tasks if in_deg[t]==0], key=lambda t: priority[t])
    result = []
    while ready:
        node = ready.pop(0); result.append(node)
        for s in succ[node]:
            in_deg[s] -= 1
            if in_deg[s]==0:
                ready.append(s); ready.sort(key=lambda x: priority[x])
    return result

def ga_simple_nsga2(tasks, vms, pop_size=80, n_gen=200,
                    cx_rate=0.9, mut_rate=0.15, seed=42, verbose=True):
    random.seed(seed); vm_names = list(vms.keys())
    def rand_chrom():
        return random_topo_order(tasks), {t: random.choice(vm_names) for t in tasks}
    def cx_chrom(p1, p2):
        o1,v1=p1; o2,v2=p2
        if random.random()>=cx_rate: return copy.deepcopy(p1), copy.deepcopy(p2)
        co1=crossover_order_simple(o1,o2,tasks); co2=crossover_order_simple(o2,o1,tasks)
        cv1,cv2={},{}
        for t in v1:
            if random.random()<0.5: cv1[t],cv2[t]=v1[t],v2[t]
            else:                    cv1[t],cv2[t]=v2[t],v1[t]
        return (co1,cv1),(co2,cv2)
    def mut_chrom(chrom):
        order,vm_map=chrom
        return mutate_order_simple(order,tasks,mut_rate), {t:(random.choice(vm_names) if random.random()<mut_rate else vm_map[t]) for t in vm_map}
    def get_obj(chrom):
        o,v=chrom; m,c,e,_=evaluate(o,v,tasks,vms); return (m,c,e)
    pop=[rand_chrom() for _ in range(pop_size)]; obj=[get_obj(p) for p in pop]
    if verbose:
        print(f"\n  GA Simple  {'Gen':>5}  {'|F0|':>6}  {'MinMksp':>10}  {'MinCost':>10}  {'MinEnrg':>10}")
        print("  "+"─"*58)
    for gen in range(n_gen):
        fronts=fast_non_dominated_sort(obj); ranks=[0]*pop_size; dists={}
        for rank,front in enumerate(fronts):
            for idx in front: ranks[idx]=rank
            cd=crowding_distance_standard(front,obj); dists.update(cd)
        children=[]; ch_obj=[]
        while len(children)<pop_size:
            i1=binary_tournament(pop_size,ranks,dists); i2=binary_tournament(pop_size,ranks,dists)
            c1,c2=cx_chrom(pop[i1],pop[i2]); c1=mut_chrom(c1); c2=mut_chrom(c2)
            children.append(c1); ch_obj.append(get_obj(c1))
            if len(children)<pop_size: children.append(c2); ch_obj.append(get_obj(c2))
        pop,obj=select_survivors(pop+children,obj+ch_obj,pop_size,crowding_distance_standard)
        if verbose and (gen%40==0 or gen==n_gen-1):
            pf=fronts[0]; po=[obj[i] for i in pf]
            print(f"  GA Simple  {gen:>5}  {len(pf):>6}  {min(o[0] for o in po):>10.2f}  {min(o[1] for o in po):>10.4f}  {min(o[2] for o in po):>10.2f}")
    pf_idx=fast_non_dominated_sort(obj)[0]
    return [pop[i] for i in pf_idx], [obj[i] for i in pf_idx]

# ══════════════════════════════════════════════════════════════════════
#  8. ALGO 2 — HYBRID GA+HEFT NSGA-II
# ══════════════════════════════════════════════════════════════════════

def ga_heft_nsga2(tasks, vms, pop_size=80, n_gen=200, cx_rate=0.9,
                  mut_rate=0.15, heft_ratio=0.3, ls_rate=0.4, seed=42, verbose=True):
    random.seed(seed); vm_names=list(vms.keys())
    groups_ref=build_groups(tasks); ref=compute_ref(tasks,vms,groups_ref,n_samples=300)
    def rand_chrom():
        rg=[random.sample(g,len(g)) for g in groups_ref]
        return rg, {t: random.choice(vm_names) for t in tasks}
    def get_obj(groups, vm_map):
        order=groups_to_order(groups); m,c,e,_=evaluate(order,vm_map,tasks,vms); return (m,c,e)
    def _ox(a,b,cut):
        seg=a[:cut]; return seg+[t for t in b if t not in set(seg)]
    def cx(p1,p2):
        o1,v1=p1; o2,v2=p2
        if random.random()>=cx_rate: return copy.deepcopy(p1), copy.deepcopy(p2)
        c1o,c2o=[],[]
        for l1,l2 in zip(o1,o2):
            n=len(l1)
            if n<=1: c1o.append(l1[:]); c2o.append(l2[:])
            else:
                pt=random.randint(1,n-1); c1o.append(_ox(l1,l2,pt)); c2o.append(_ox(l2,l1,pt))
        c1v,c2v={},{}
        for t in v1:
            if random.random()<0.5: c1v[t],c2v[t]=v1[t],v2[t]
            else:                    c1v[t],c2v[t]=v2[t],v1[t]
        return (c1o,c1v),(c2o,c2v)
    def mut(chrom):
        groups,vm_map=chrom; new_g=[]
        for g in groups:
            ng=g[:]
            if len(ng)>1:
                for i in range(len(ng)):
                    if random.random()<mut_rate:
                        j=random.randint(0,len(ng)-1); ng[i],ng[j]=ng[j],ng[i]
            new_g.append(ng)
        return new_g, {t:(random.choice(vm_names) if random.random()<mut_rate else vm_map[t]) for t in vm_map}
    heft_g,heft_v=heft_schedule(tasks,vms)
    pop=[(heft_g, copy.deepcopy(heft_v))]
    n_heft=max(1,int(pop_size*heft_ratio))-1
    for _ in range(n_heft): pop.append(([g[:] for g in heft_g],{t:random.choice(vm_names) for t in tasks}))
    while len(pop)<pop_size: pop.append(rand_chrom())
    obj=[get_obj(*p) for p in pop]
    if verbose:
        print(f"\n  GA+HEFT    {'Gen':>5}  {'|F0|':>6}  {'MinMksp':>10}  {'MinCost':>10}  {'MinEnrg':>10}")
        print("  "+"─"*58)
    for gen in range(n_gen):
        fronts=fast_non_dominated_sort(obj); ranks=[0]*pop_size; dists={}
        for rank,front in enumerate(fronts):
            for idx in front: ranks[idx]=rank
            cd=crowding_distance_weighted(front,obj,ref); dists.update(cd)
        children=[]; ch_obj=[]
        while len(children)<pop_size:
            i1=binary_tournament(pop_size,ranks,dists); i2=binary_tournament(pop_size,ranks,dists)
            c1,c2=cx(pop[i1],pop[i2]); c1=mut(c1); c2=mut(c2)
            if random.random()<ls_rate: c1=heft_local_search(*c1,tasks,vms)
            if random.random()<ls_rate: c2=heft_local_search(*c2,tasks,vms)
            children.append(c1); ch_obj.append(get_obj(*c1))
            if len(children)<pop_size: children.append(c2); ch_obj.append(get_obj(*c2))
        pop,obj=select_survivors(pop+children,obj+ch_obj,pop_size,crowding_distance_weighted,ref=ref)
        if verbose and (gen%40==0 or gen==n_gen-1):
            pf=fronts[0]; po=[obj[i] for i in pf]
            print(f"  GA+HEFT    {gen:>5}  {len(pf):>6}  {min(o[0] for o in po):>10.2f}  {min(o[1] for o in po):>10.4f}  {min(o[2] for o in po):>10.2f}")
    pf_idx=fast_non_dominated_sort(obj)[0]
    pf_chroms=[pop[i] for i in pf_idx]; pf_obj=[obj[i] for i in pf_idx]
    pf_std=[(groups_to_order(g),v) for g,v in pf_chroms]
    return pf_std, pf_obj

# ══════════════════════════════════════════════════════════════════════
#  9. ALGO 3 — MOEA/D (Tchebycheff)
#  Encoding identical to Simple GA: topological order + vm_map
#  Uniform 3D weight vectors, neighborhood T, external archive
# ══════════════════════════════════════════════════════════════════════

def _moead_weights(H):
    """Uniform 3D weight vectors. H=divisions → ~C(H+2,2) vectors."""
    W = []
    for i in range(H+1):
        for j in range(H+1-i):
            k = H-i-j; W.append((i/H, j/H, k/H))
    return W

def _moead_neighbours(W, T):
    """T nearest neighbours by Euclidean distance between vectors."""
    N = len(W)
    return [sorted(range(N), key=lambda j: sum((W[i][d]-W[j][d])**2 for d in range(3)))[:T]
            for i in range(N)]

def _trim_archive(arch_sol, arch_obj, max_size):
    """
    Trims the archive to max_size solutions to ensure a fair
    comparison with the GAs (same number of solutions in the front).
    Step 1: keep only non-dominated solutions.
    Step 2: if still too large, sort by crowding distance ↓
            and truncate → preserves maximum diversity.
    """
    if len(arch_obj) <= max_size:
        return arch_sol, arch_obj

    # Step 1: non-dominated filter
    nd_idx = []
    for i, p in enumerate(arch_obj):
        dom = False
        for j, q in enumerate(arch_obj):
            if i == j: continue
            if dominates(list(q), list(p)): dom = True; break
        if not dom: nd_idx.append(i)
    nd_sol = [arch_sol[i] for i in nd_idx]
    nd_obj = [arch_obj[i] for i in nd_idx]

    if len(nd_obj) <= max_size:
        return nd_sol, nd_obj

    # Step 2: crowding distance → keep the most spread out
    n = len(nd_obj); cd = [0.0] * n
    for m in range(3):
        idx_s = sorted(range(n), key=lambda i: nd_obj[i][m])
        cd[idx_s[0]] = cd[idx_s[-1]] = float("inf")
        lo = nd_obj[idx_s[0]][m]; hi = nd_obj[idx_s[-1]][m]
        rng = hi - lo + 1e-9
        for k in range(1, n-1):
            cd[idx_s[k]] += (nd_obj[idx_s[k+1]][m] - nd_obj[idx_s[k-1]][m]) / rng
    ranked = sorted(range(n), key=lambda i: cd[i], reverse=True)
    kept   = sorted(ranked[:max_size])
    return [nd_sol[i] for i in kept], [nd_obj[i] for i in kept]


def moead_nsga2(tasks, vms, H=8, T=5, n_gen=200, mut_rate=0.05,
                max_archive=None, seed=42, verbose=True):
    """
    MOEA/D Tchebycheff:
      g(x|λ,z*) = max_i { λ_i · |f_i(x) − z*_i| }

    max_archive: maximum size of the final archive (None = unlimited).
                 Set to the GAs' pop_size for a fair comparison.
    Returns (pf_chroms, pf_obj) — same format as ga_simple_nsga2.
    """
    random.seed(seed)
    vm_names   = list(vms.keys())
    groups_ref = build_groups(tasks)
    W = _moead_weights(H); N = len(W); B = _moead_neighbours(W, T)

    # ── Genetic operators ────────────────────────────────────────────
    def rand_sol():
        rg    = [random.sample(g, len(g)) for g in groups_ref]
        order = groups_to_order(rg)
        return order, {t: random.choice(vm_names) for t in tasks}

    def crossover_m(s1, s2):
        o1,vm1=s1; o2,vm2=s2; n=len(o1); pt=random.randint(1,n-2)
        seg=o1[:pt]; seg_s=set(seg)
        child_order = seg + [t for t in o2 if t not in seg_s]
        child_order = repair_order_kahn(child_order, tasks)
        child_vm = {t: (vm1[t] if random.random()<0.5 else vm2[t]) for t in tasks}
        return child_order, child_vm

    def mutate_m(sol):
        order, vm_map = list(sol[0]), dict(sol[1])
        if random.random() < mut_rate and len(order) > 1:
            i, j = random.sample(range(len(order)), 2)
            order[i], order[j] = order[j], order[i]
            order = repair_order_kahn(order, tasks)
        for t in list(vm_map.keys()):
            if random.random() < mut_rate:
                vm_map[t] = random.choice(vm_names)
        return order, vm_map

    def tcheby(obj, w, z):
        return max(w[m]*abs(obj[m]-z[m]) for m in range(3))

    # ── Initialisation ───────────────────────────────────────────────
    pop  = [rand_sol() for _ in range(N)]
    objs = []
    for s in pop:
        m, c, e, _ = evaluate(s[0], s[1], tasks, vms)
        objs.append((m, c, e))

    z_star = [min(objs[i][m] for i in range(N)) for m in range(3)]

    # Non-dominated external archive
    arch_obj = list(objs); arch_sol = list(pop)

    def update_archive(new_sol, new_obj):
        to_remove = []
        for k, ao in enumerate(arch_obj):
            if dominates(list(new_obj), list(ao)):
                to_remove.append(k)
            elif dominates(list(ao), list(new_obj)):
                return
        for k in sorted(to_remove, reverse=True):
            arch_obj.pop(k); arch_sol.pop(k)
        arch_obj.append(new_obj); arch_sol.append(new_sol)

    if verbose:
        print(f"\n  MOEA/D     {'Gen':>5}  {'|Arch|':>7}  "
              f"{'z*[0]':>10}  {'z*[1]':>10}  {'z*[2]':>10}")
        print("  " + "─"*58)

    # ── Main loop ────────────────────────────────────────────────────
    for gen in range(n_gen):
        for i in range(N):
            p1i, p2i = random.sample(B[i], 2)
            child = crossover_m(pop[p1i], pop[p2i])
            child = mutate_m(child)
            m, c, e, _ = evaluate(child[0], child[1], tasks, vms)
            c_obj = (m, c, e)
            # Update z*
            for d in range(3):
                if c_obj[d] < z_star[d]: z_star[d] = c_obj[d]
            # Update neighbours
            for j in B[i]:
                if tcheby(c_obj, W[j], z_star) <= tcheby(objs[j], W[j], z_star):
                    pop[j] = child; objs[j] = c_obj
            update_archive(child, c_obj)

        # Periodic pruning if archive is too large
        if max_archive and len(arch_obj) > max_archive * 2:
            arch_sol, arch_obj = _trim_archive(arch_sol, arch_obj, max_archive)

        if verbose and (gen % 40 == 0 or gen == n_gen-1):
            print(f"  MOEA/D     {gen:>5}  {len(arch_obj):>7}  "
                  f"{z_star[0]:>10.2f}  {z_star[1]:>10.4f}  {z_star[2]:>10.2f}")

    # Final pruning to max_archive solutions
    if max_archive:
        arch_sol, arch_obj = _trim_archive(arch_sol, arch_obj, max_archive)

    return arch_sol, arch_obj

# ══════════════════════════════════════════════════════════════════════
#  10. METRICS: HV & IGD+
# ══════════════════════════════════════════════════════════════════════

def build_global_ref(tasks, vms, groups_ref, n_samples=500, seed=999):
    vm_names=list(vms.keys()); rng=random.Random(seed)
    all_m,all_c,all_e=[],[],[]
    for vm in vm_names:
        order=groups_to_order(groups_ref); vm_map={t:vm for t in tasks}
        m,c,e,_=evaluate(order,vm_map,tasks,vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    for _ in range(n_samples):
        rg=[rng.sample(g,len(g)) for g in groups_ref]; order=groups_to_order(rg)
        vm_map={t:rng.choice(vm_names) for t in tasks}
        m,c,e,_=evaluate(order,vm_map,tasks,vms)
        all_m.append(m); all_c.append(c); all_e.append(e)
    bounds={"m_min":min(all_m),"m_max":max(all_m),"c_min":min(all_c),
            "c_max":max(all_c),"e_min":min(all_e),"e_max":max(all_e)}
    for k in ["m","c","e"]:
        if bounds[f"{k}_max"]-bounds[f"{k}_min"]<1e-9: bounds[f"{k}_max"]=bounds[f"{k}_min"]+1.0
    return bounds

def normalize_to_unit(obj_list, bounds):
    def _n(m,c,e):
        f1=max(0.0,(m-bounds["m_min"])/(bounds["m_max"]-bounds["m_min"]))
        f2=max(0.0,(c-bounds["c_min"])/(bounds["c_max"]-bounds["c_min"]))
        f3=max(0.0,(e-bounds["e_min"])/(bounds["e_max"]-bounds["e_min"]))
        return (min(f1,1.0),min(f2,1.0),min(f3,1.0))
    return [_n(*o) for o in obj_list]

def _pareto_nd(pts):
    nd=[]
    for p in pts:
        if not any(all(qi<=pi for qi,pi in zip(q,p)) and
                   any(qi<pi for qi,pi in zip(q,p)) for q in pts if q is not p):
            nd.append(p)
    return nd

def build_true_ref_front(tasks, vms, groups_ref, bounds, n_samples=2000, seed=888):
    vm_names=list(vms.keys()); rng=random.Random(seed); all_pts=[]
    for _ in range(n_samples):
        rg=[rng.sample(g,len(g)) for g in groups_ref]; order=groups_to_order(rg)
        vm_map={t:rng.choice(vm_names) for t in tasks}
        m,c,e,_=evaluate(order,vm_map,tasks,vms); all_pts.append((m,c,e))
    return _pareto_nd(normalize_to_unit(all_pts,bounds))

def hypervolume_3d(norm_obj, ref_point=(1.1,1.1,1.1)):
    ref=ref_point
    pts=[o for o in norm_obj if o[0]<ref[0] and o[1]<ref[1] and o[2]<ref[2]]
    if not pts: return 0.0
    pts=_pareto_nd(pts); pts_sorted=sorted(pts,key=lambda p:p[2],reverse=True)
    prev_z=ref[2]; hv=0.0; active=[]
    for p in pts_sorted:
        z_slice=prev_z-p[2]; active.append((p[0],p[1]))
        act_s=sorted(set(active),key=lambda a:a[0])
        area=0.0; prev_y=ref[1]
        for (ax,ay) in act_s:
            if ay<prev_y: area+=(ref[0]-ax)*(prev_y-ay); prev_y=ay
        hv+=area*z_slice; prev_z=p[2]
    return hv

def igd_plus(norm_front_A, norm_ref_front_B):
    if not norm_ref_front_B or not norm_front_A: return float("inf")
    total=0.0
    for b in norm_ref_front_B:
        min_d=min(math.sqrt(sum(max(0.0,a[i]-b[i])**2 for i in range(len(b)))) for a in norm_front_A)
        total+=min_d
    return total/len(norm_ref_front_B)

# ══════════════════════════════════════════════════════════════════════
#  11. VISUALIZATIONS — 3 algorithms
# ══════════════════════════════════════════════════════════════════════

def plot_pareto_fronts_3(obj1, obj2, obj3,
                         save_path="pareto_cybershake.png"):
    m1=[o[0] for o in obj1]; c1=[o[1] for o in obj1]; e1=[o[2] for o in obj1]
    m2=[o[0] for o in obj2]; c2=[o[1] for o in obj2]; e2=[o[2] for o in obj2]
    m3=[o[0] for o in obj3]; c3=[o[1] for o in obj3]; e3=[o[2] for o in obj3]
    fig=plt.figure(figsize=(16,13))
    fig.suptitle("Pareto Fronts — NSGA-II simple  vs NSGA-II+HEFT vs MOEA/D\nCyberShake 50 tasks × 5 VMs",
                 fontsize=14,fontweight="bold",y=0.98)
    configs=[(221,(m1,c1,m2,c2,m3,c3),"Makespan (s)","Cost ($)","Makespan vs Cost"),
             (222,(m1,e1,m2,e2,m3,e3),"Makespan (s)","Energy (kj)","Makespan vs Energy"),
             (223,(c1,e1,c2,e2,c3,e3),"Cost ($)","Energy (kj)","Cost vs Energy")]
    for sid,(x1,y1_,x2,y2_,x3,y3_),xl,yl,title in configs:
        ax=fig.add_subplot(sid)
        ax.scatter(x1,y1_,c="royalblue",marker="o",s=60,alpha=0.8,label=f"NSGA-II  Simple ({len(obj1)} pts)",zorder=3)
        ax.scatter(x2,y2_,c="tomato",   marker="^",s=60,alpha=0.8,label=f"NSGA-II +HEFT ({len(obj2)} pts)",  zorder=3)
        ax.scatter(x3,y3_,c="#1D9E75",  marker="s",s=60,alpha=0.8,label=f"MOEA/D ({len(obj3)} pts)",   zorder=3)
        for x,y,col in [(x1,y1_,"royalblue"),(x2,y2_,"tomato"),(x3,y3_,"#1D9E75")]:
            if x: s=sorted(zip(x,y)); ax.plot([p[0] for p in s],[p[1] for p in s],col,lw=0.8,alpha=0.35)
        ax.set_xlabel(xl,fontsize=10); ax.set_ylabel(yl,fontsize=10)
        ax.set_title(title,fontsize=11,fontweight="bold"); ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
    ax4=fig.add_subplot(224,projection="3d")
    ax4.scatter(m1,c1,e1,c="royalblue",marker="o",s=40,alpha=0.8,label="NSGA-II simple")
    ax4.scatter(m2,c2,e2,c="tomato",   marker="^",s=40,alpha=0.8,label="NSGA-II +HEFT")
    ax4.scatter(m3,c3,e3,c="#1D9E75",  marker="s",s=40,alpha=0.8,label="MOEA/D")
    ax4.set_xlabel("Makespan",fontsize=9); ax4.set_ylabel("Cost",fontsize=9)
    ax4.set_zlabel("Energy",fontsize=9); ax4.set_title("3D View",fontsize=11,fontweight="bold")
    ax4.legend(fontsize=9)
    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close()
    print(f"  → Pareto plot saved: {save_path}")

def plot_runs_evolution_3(ga_hvs, heft_hvs, moead_hvs,
                          ga_igds, heft_igds, moead_igds,
                          save_path="runs_evolution_cybershake.png"):
    n=len(ga_hvs); runs=list(range(1,n+1)); avg=lambda l: sum(l)/len(l)
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5))
    fig.suptitle("Evolution per run — NSGA-II  Simple vs NSGA-II +HEFT vs MOEA/D\nCyberShake 50 tasks × 5 VMs",
                 fontsize=13,fontweight="bold")
    for vals,col,mk,lbl in [(ga_hvs,"royalblue","o","NSGA-II Simple"),
                             (heft_hvs,"tomato","^","NSGA-II+HEFT"),
                             (moead_hvs,"#1D9E75","s","MOEA/D")]:
        ax1.plot(runs,vals,f"{mk}-",color=col,lw=1.8,ms=5,label=f"{lbl} (moy={avg(vals):.4f})")
        ax1.axhline(avg(vals),color=col,lw=1.2,ls="--",alpha=0.7)
    ax1.set_xlabel("Run number",fontsize=10); ax1.set_ylabel("Normalized HV ∈ [0,1.331]",fontsize=10)
    ax1.set_title("HV per run  (↑ better)",fontsize=11,fontweight="bold")
    ax1.set_xticks(runs); ax1.legend(fontsize=8); ax1.grid(True,alpha=0.3)
    for vals,col,mk,lbl in [(ga_igds,"royalblue","o","NSGA-II Simple"),
                             (heft_igds,"tomato","^","NSGA-II+HEFT"),
                             (moead_igds,"#1D9E75","s","MOEA/D")]:
        ax2.plot(runs,vals,f"{mk}-",color=col,lw=1.8,ms=5,label=f"{lbl} (moy={avg(vals):.5f})")
        ax2.axhline(avg(vals),color=col,lw=1.2,ls="--",alpha=0.7)
    ax2.set_xlabel("Run number",fontsize=10); ax2.set_ylabel("IGD+",fontsize=10)
    ax2.set_title("IGD+ per run  (↓ better)",fontsize=11,fontweight="bold")
    ax2.set_xticks(runs); ax2.legend(fontsize=8); ax2.grid(True,alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close()
    print(f"  → Runs plot saved: {save_path}")

def plot_metrics_bar_3(metrics_dict,
                       save_path="metrics_bar_cybershake.png"):
    labels=list(metrics_dict.keys())
    metric_names=["HV (↑ better)","IGD+ (↓ better)","|Pareto|"]
    colors_bar=["royalblue","tomato","#1D9E75"]
    fig,axes=plt.subplots(1,3,figsize=(16,5))
    for ax,mn in zip(axes,metric_names):
        idx=metric_names.index(mn)
        vals=[metrics_dict[lab][idx] for lab in labels]
        bars=ax.bar(labels,vals,color=colors_bar[:len(labels)],alpha=0.85,edgecolor="black",width=0.45)
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+max(abs(v) for v in vals)*0.01,
                    f"{val:.4f}",ha="center",va="bottom",fontsize=10,fontweight="bold")
        ax.set_title(mn,fontsize=11,fontweight="bold")
        ax.set_ylabel("Mean value (20 runs)"); ax.grid(axis="y",alpha=0.3)
    fig.suptitle("Pareto Metrics — Mean of 20 runs\nCyberShake 50 tasks",fontsize=13,fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path,dpi=150,bbox_inches="tight"); plt.close()
    print(f"  → Metrics plot saved: {save_path}")

# ══════════════════════════════════════════════════════════════════════
#  12. TEXT DISPLAY
# ══════════════════════════════════════════════════════════════════════

def display_pareto(label, pf_chroms, pf_obj, tasks, vms, max_show=8):
    W=72; print(f"\n  {'═'*W}")
    print(f"  PARETO FRONT — {label}  ({len(pf_obj)} non-dominated solutions)")
    print(f"  {'═'*W}")
    print(f"  {'#':>4}  {'Makespan (s)':>13}  {'Cost ($)':>10}  {'Energy (kJ)':>14}  Profile")
    print(f"  {'─'*W}")
    min_m=min(o[0] for o in pf_obj); min_c=min(o[1] for o in pf_obj); min_e=min(o[2] for o in pf_obj)
    si=sorted(range(len(pf_obj)),key=lambda i: pf_obj[i][0])
    for rank,i in enumerate(si[:max_show]):
        m,c,e=pf_obj[i]; tags=[]
        if abs(m-min_m)<1e-4: tags.append("★ makespan")
        if abs(c-min_c)<1e-9: tags.append("★ cost")
        if abs(e-min_e)<1e-4: tags.append("★ energy")
        print(f"  {rank+1:>4}  {m:>13.2f}  {c:>10.4f}  {e:>14.2f}  {'  '.join(tags)}")
    if len(pf_obj)>max_show: print(f"  ... {len(pf_obj)-max_show} more solutions")
    print(f"  {'─'*W}")
    extremes=[("Best Makespan",min(range(len(pf_obj)),key=lambda i:pf_obj[i][0])),
              ("Best Cost",    min(range(len(pf_obj)),key=lambda i:pf_obj[i][1])),
              ("Best Energy",min(range(len(pf_obj)),key=lambda i:pf_obj[i][2]))]
    seen=set()
    for lbl,idx in extremes:
        if idx in seen: continue
        seen.add(idx)
        order,vm_map=pf_chroms[idx]; m,c,e,sched=evaluate(order,vm_map,tasks,vms)
        print(f"\n  ── {lbl}: Mksp={m:.2f}s  Cost={c:.4f}$  Enrg={e:.2f}kWh")
        vm_load={v:0 for v in vms}
        for t in order: vm_load[vm_map[t]]+=1
        print(f"     VM load: "+"  ".join(f"{v}={vm_load[v]}" for v in vms))
    print(f"  {'═'*W}")

def display_metrics_3(obj1, obj2, obj3, bounds, true_ref_front):
    norm1=normalize_to_unit(obj1,bounds); norm2=normalize_to_unit(obj2,bounds)
    norm3=normalize_to_unit(obj3,bounds)
    hv1=hypervolume_3d(norm1); hv2=hypervolume_3d(norm2); hv3=hypervolume_3d(norm3)
    igd1=igd_plus(norm1,true_ref_front); igd2=igd_plus(norm2,true_ref_front)
    igd3=igd_plus(norm3,true_ref_front)
    W=76; print("\n"+"═"*W); print("  PARETO EVALUATION METRICS — 3 algorithms"); print("─"*W)
    print(f"  HV reference point (fixed, normalized): (1.1, 1.1, 1.1)")
    print(f"  Reference front P* (independent)       : {len(true_ref_front)} solutions")
    print(f"  Normalized HV ∈ [0, 1.331]  — fair comparison")
    print("─"*W)
    print(f"  {'Metric':<28} {'NSGA-II Simple':>14}  {'NSGA-II+HEFT':>14}  {'MOEA/D':>14}")
    print("─"*W)
    def best3(v1,v2,v3,hi=True):
        best=max(v1,v2,v3) if hi else min(v1,v2,v3)
        return ["◄" if abs(v-best)<1e-12 else "  " for v in [v1,v2,v3]]
    b=best3(hv1,hv2,hv3,hi=True)
    print(f"  {'HV (↑ better)':<28} {hv1:>12.6f}{b[0]}  {hv2:>12.6f}{b[1]}  {hv3:>12.6f}{b[2]}")
    b=best3(igd1,igd2,igd3,hi=False)
    print(f"  {'IGD+ (↓ better)':<28} {igd1:>12.6f}{b[0]}  {igd2:>12.6f}{b[1]}  {igd3:>12.6f}{b[2]}")
    b=best3(len(obj1),len(obj2),len(obj3),hi=True)
    print(f"  {'|Pareto| (↑ better)':<28} {len(obj1):>14}{b[0]}  {len(obj2):>14}{b[1]}  {len(obj3):>14}{b[2]}")
    print("─"*W)
    for lbl,objs in [("Min Makespan (s)",0),("Min Cost ($)",1),("Min Energy (kj)",2)]:
        print(f"  {lbl:<28} {min(o[objs] for o in obj1):>14.4f}   {min(o[objs] for o in obj2):>14.4f}   {min(o[objs] for o in obj3):>14.4f}")
    print("═"*W)
    return hv1,hv2,hv3,igd1,igd2,igd3

# ══════════════════════════════════════════════════════════════════════
#  13. MAIN
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    tasks=cybershake_50(); vms=vms_5(); N_RUNS=20  # reduced for testing

    print("╔"+"═"*72+"╗")
    print("║  NSGA-II: NSGA-II Simple  vs  NSGA-II+HEFT Hybrid  vs  MOEA/D"+" "*16+"║")
    print(f"║  CyberShake {len(tasks)} tasks × {len(vms)} VMs  |  {N_RUNS} runs  |  seed=42"+" "*19+"║")
    print("╚"+"═"*72+"╝")

    groups_ref=build_groups(tasks)
    print("\n  Computing global bounds...")
    bounds=build_global_ref(tasks,vms,groups_ref,n_samples=300,seed=999)
    print(f"  Makespan:[{bounds['m_min']:.1f},{bounds['m_max']:.1f}]  Cost:[{bounds['c_min']:.4f},{bounds['c_max']:.4f}]  Energy:[{bounds['e_min']:.1f},{bounds['e_max']:.1f}]")
    print("\n  Building reference front P*...")
    true_ref_front=build_true_ref_front(tasks,vms,groups_ref,bounds,n_samples=500,seed=888)
    print(f"  P*: {len(true_ref_front)} solutions")

    ga_hvs=[]; ga_igds=[]; ga_nf=[]
    heft_hvs=[]; heft_igds=[]; heft_nf=[]
    moead_hvs=[]; moead_igds=[]; moead_nf=[]

    avg=lambda l: sum(l)/len(l)
    std=lambda l,m: math.sqrt(sum((x-m)**2 for x in l)/len(l))

    print("\n"+"═"*82)
    print(f"  {N_RUNS} RUNS — NSGA-II  Simple  vs  NSGA-II +HEFT  vs  MOEA/D")
    print("═"*82)
    print(f"  {'Run':>4}  {'GA HV':>10} {'GA IGD+':>10} {'GA|F|':>6}  "
          f"{'HB HV':>10} {'HB IGD+':>10} {'HB|F|':>6}  "
          f"{'MD HV':>10} {'MD IGD+':>10} {'MD|F|':>6}")
    print("  "+"─"*78)

    for run in range(N_RUNS):
        pf1c,pf1o=ga_simple_nsga2(tasks,vms,pop_size=70,n_gen=350,cx_rate=0.8,mut_rate=0.05,seed=run,verbose=False)
        pf2c,pf2o=ga_heft_nsga2(tasks,vms,pop_size=70,n_gen=350,cx_rate=0.8,mut_rate=0.05,heft_ratio=0.3,ls_rate=0.5,seed=run,verbose=False)
        pf3c,pf3o=moead_nsga2(tasks,vms,H=7,T=5,n_gen=350,mut_rate=0.05,max_archive=70,seed=run,verbose=False)

        norm1=normalize_to_unit(pf1o,bounds); norm2=normalize_to_unit(pf2o,bounds)
        norm3=normalize_to_unit(pf3o,bounds)
        hv1=hypervolume_3d(norm1); hv2=hypervolume_3d(norm2); hv3=hypervolume_3d(norm3)
        i1=igd_plus(norm1,true_ref_front); i2=igd_plus(norm2,true_ref_front); i3=igd_plus(norm3,true_ref_front)

        ga_hvs.append(hv1); ga_igds.append(i1); ga_nf.append(len(pf1o))
        heft_hvs.append(hv2); heft_igds.append(i2); heft_nf.append(len(pf2o))
        moead_hvs.append(hv3); moead_igds.append(i3); moead_nf.append(len(pf3o))
        print(f"  {run+1:>4}  {hv1:>10.6f} {i1:>10.6f} {len(pf1o):>6}  "
              f"{hv2:>10.6f} {i2:>10.6f} {len(pf2o):>6}  "
              f"{hv3:>10.6f} {i3:>10.6f} {len(pf3o):>6}")

    print("  "+"─"*78)
    print(f"  {'Moy':>4}  {avg(ga_hvs):>10.6f} {avg(ga_igds):>10.6f} {avg(ga_nf):>6.1f}  "
          f"{avg(heft_hvs):>10.6f} {avg(heft_igds):>10.6f} {avg(heft_nf):>6.1f}  "
          f"{avg(moead_hvs):>10.6f} {avg(moead_igds):>10.6f} {avg(moead_nf):>6.1f}")
    print(f"  {'Std':>4}  {std(ga_hvs,avg(ga_hvs)):>10.6f} {std(ga_igds,avg(ga_igds)):>10.6f} {'—':>6}  "
          f"{std(heft_hvs,avg(heft_hvs)):>10.6f} {std(heft_igds,avg(heft_igds)):>10.6f} {'—':>6}  "
          f"{std(moead_hvs,avg(moead_hvs)):>10.6f} {std(moead_igds,avg(moead_igds)):>10.6f} {'—':>6}")
    print("═"*82)

    # Summary
    print("\n"+"═"*82)
    print("  STATISTICAL SUMMARY")
    print("═"*82)
    print(f"  {'Algorithm':<22} {'HV moy':>12} {'HV std':>10} {'IGD+ moy':>12} {'IGD+ std':>10} {'|F0| moy':>9}")
    print("─"*82)
    for name,hvs,igds,nf in [("NSGA-II Simple",ga_hvs,ga_igds,ga_nf),
                               ("NSGA-II+HEFT Hybrid",heft_hvs,heft_igds,heft_nf),
                               ("MOEA/D",moead_hvs,moead_igds,moead_nf)]:
        print(f"  {name:<22} {avg(hvs):>12.6f} {std(hvs,avg(hvs)):>10.6f} {avg(igds):>12.6f} {std(igds,avg(igds)):>10.6f} {avg(nf):>9.1f}")
    print("─"*82)
    for name_a,hvsa,igdsa,name_b,hvsb,igdsb in [
        ("NSGA-II+HEFT",heft_hvs,heft_igds,"NSGA-II Simple",ga_hvs,ga_igds),
        ("MOEA/D", moead_hvs,moead_igds,"NSGA-II Simple",ga_hvs,ga_igds)]:
        dh=avg(hvsb)-avg(hvsa); di=avg(igdsa)-avg(igdsb)
        print(f"  Δ {name_a}−{name_b} : HV={dh:+.6f} {'✓' if dh>0 else '✗'}  IGD+={di:+.6f} {'✓' if di<0 else '✗'}")
    print("═"*82)

    # Main run seed=42
    print("\n\n╔"+"═"*72+"╗"); print("║  MAIN RUN — seed=42"+" "*47+"║"); print("╚"+"═"*72+"╝")
    print("\n  [1/3] NSGA-II Simple (seed=42)")
    t0=time.time(); pf1_c,pf1_o=ga_simple_nsga2(tasks,vms,pop_size=70,n_gen=350,seed=42,verbose=True); t1=time.time()
    print(f"  Time: {t1-t0:.1f}s  |  |Pareto|={len(pf1_o)}")
    print("\n  [2/3] NSGA-II+HEFT (seed=42)")
    t2=time.time(); pf2_c,pf2_o=ga_heft_nsga2(tasks,vms,pop_size=70,n_gen=350,heft_ratio=0.3,ls_rate=0.5,seed=42,verbose=True); t3=time.time()
    print(f"  Time: {t3-t2:.1f}s  |  |Pareto|={len(pf2_o)}")
    print("\n  [3/3] MOEA/D (seed=42)")
    t4=time.time(); pf3_c,pf3_o=moead_nsga2(tasks,vms,H=7,T=5,n_gen=350,mut_rate=0.05,max_archive=70,seed=42,verbose=True); t5=time.time()
    print(f"  Time: {t5-t4:.1f}s  |  |Pareto|={len(pf3_o)}")

    display_metrics_3(pf1_o,pf2_o,pf3_o,bounds,true_ref_front)
    display_pareto("NSGA-II Simple (seed=42)",   pf1_c,pf1_o,tasks,vms)
    display_pareto("NSGA-II+HEFT  (seed=42)",    pf2_c,pf2_o,tasks,vms)
    display_pareto("MOEA/D   (seed=42)",    pf3_c,pf3_o,tasks,vms)

    print("\n  Generating plots...")
    plot_pareto_fronts_3(pf1_o,pf2_o,pf3_o)
    plot_metrics_bar_3({
        "NSGA-II Simple":(avg(ga_hvs),avg(ga_igds),avg(ga_nf)),
        "NSGA-II+HEFT":  (avg(heft_hvs),avg(heft_igds),avg(heft_nf)),
        "MOEA/D":   (avg(moead_hvs),avg(moead_igds),avg(moead_nf)),
    })
    plot_runs_evolution_3(ga_hvs,heft_hvs,moead_hvs,ga_igds,heft_igds,moead_igds)
    print("\n  ✓ Execution completed."); print("═"*82)

╔════════════════════════════════════════════════════════════════════════╗
║  NSGA-II: NSGA-II Simple  vs  NSGA-II+HEFT Hybrid  vs  MOEA/D                ║
║  CyberShake 50 tasks × 5 VMs  |  20 runs  |  seed=42                   ║
╚════════════════════════════════════════════════════════════════════════╝

  Computing global bounds...
  Makespan:[6198.3,42200.0]  Cost:[0.7033,1.2821]  Energy:[2599.5,2954.0]

  Building reference front P*...
  P*: 47 solutions

══════════════════════════════════════════════════════════════════════════════════
  20 RUNS — NSGA-II  Simple  vs  NSGA-II +HEFT  vs  MOEA/D
══════════════════════════════════════════════════════════════════════════════════
   Run       GA HV    GA IGD+  GA|F|       HB HV    HB IGD+  HB|F|       MD HV    MD IGD+  MD|F|
  ──────────────────────────────────────────────────────────────────────────────
     1    0.797797   0.013871     70    0.753525   0.002428     70    0.812352   0.010932     70
